<a href="https://colab.research.google.com/github/ishuu9837/Fine-tuning-Llama-3.2/blob/main/fine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.2 finance fine-tuning — explained, rerunnable notebook

This notebook trains a QLoRA adapter and compares it with the base model on independent held-out questions. Each code cell has an explanation immediately above it.

**Run order:** select a Colab CUDA GPU, run the installation cell, restart the session/runtime, then run all remaining cells from authentication onward. Add your Hugging Face read token as the Colab secret `collab` and obtain access to `meta-llama/Llama-3.2-3B-Instruct` first.

**Outputs:** a score comparison table, JSON/CSV reports, generated answers and a downloadable adapter archive. Fresh accuracy values appear only after you run evaluation. This notebook has been syntax-checked locally; full CUDA training has not been executed here.

Exact-match accuracy, token F1 and teacher-forced next-token accuracy measure different things. None alone establishes factual financial correctness. All saved outputs are cleared so old numbers cannot be mistaken for this run.


## Install the training libraries

`%pip` installs packages into the notebook's Python environment. Transformers loads the model and tokenizer; TRL provides supervised fine-tuning; PEFT adds LoRA adapters; Accelerate handles training execution; bitsandbytes provides 4-bit weights and the memory-saving optimizer. Datasets loads the examples, while Evaluate, ROUGE Score and SacreBLEU support evaluation. Pandas formats the comparison table.

TRL is pinned and the other core packages have version bounds to reduce unexpected API changes. Run this cell once, then **restart the Colab session/runtime before continuing from the next cell**. Reinstalling packages halfway through training can leave incompatible modules in memory. Use a Colab CUDA GPU runtime, preferably T4 or better; this notebook's QLoRA path does not support a Mac GPU.


In [1]:
%pip install -q "transformers>=4.56,<5" "trl==0.23.1" "peft>=0.17,<0.19" "accelerate>=1.4,<2" "bitsandbytes>=0.46,<0.49" "datasets>=3,<5" evaluate rouge_score sacrebleu pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Authenticate with Hugging Face

Llama is a gated model: accept the model's access conditions on Hugging Face first. In Colab's Secrets panel, create a secret named **`collab`**, place your Hugging Face read token there, and enable notebook access to that secret. `userdata.get` retrieves the secret without embedding it in the notebook; `login` makes it available for gated downloads.

If this cell reports a missing secret, check its spelling and notebook access. A 401/403 during model loading usually requires checking token permissions and the model-access approval. Never paste a token into a code cell or commit it to GitHub.


In [4]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("collab")
if not hf_token:
    raise ValueError("Add a Hugging Face read token to the Colab secret named collab.")
login(token=hf_token)
del hf_token


## Check the GPU and choose a supported precision

`set_seed(42)` initializes the random generators used by the training libraries. Reusing a seed helps comparisons, although different hardware and library versions can still produce different results.

The cell stops immediately if CUDA is unavailable, so an unsuitable runtime fails before a large download. `is_bf16_supported()` selects BF16 on compatible GPUs and FP16 otherwise. A Colab T4 uses FP16; forcing BF16 on it was one of the original training problems. `compute_dtype` is used for arithmetic inside the quantized model, while `use_bf16` later configures the trainer's mixed precision.


In [5]:
# Restart the Colab runtime after installing dependencies, then run from the login cell.
import torch
from transformers import set_seed
SEED = 42
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU in Colab.")
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}; compute dtype: {compute_dtype}")

GPU: Tesla T4; compute dtype: torch.bfloat16


## Load the dataset and remove duplicate questions

`load_dataset` downloads `gbharti/finance-alpaca`. Its rows have instruction, input and output fields. Empty instructions or answers are rejected. The filter builds a key from the lowercased instruction and input, collapsing repeated whitespace, and keeps only the first row for each key.

This happens **before splitting**, so identical normalized questions cannot appear in both training and testing, even when their answers differ. `load_from_cache_file=False` ensures the stateful filter rebuilds its `seen` set on each run. This catches exact normalized duplicates, not paraphrases or semantic duplicates. The dataset also includes general instruction examples, so its name does not imply every row is financial.


In [6]:
from datasets import load_dataset

dataset = load_dataset("gbharti/finance-alpaca", split="train")
# Keep each normalized question/input in just one partition, even if answers differ.
seen = set()
def unique_question(example):
    if not example["instruction"].strip() or not example["output"].strip():
        return False
    key = (" ".join(example["instruction"].lower().split()),
           " ".join(example["input"].lower().split()))
    if key in seen:
        return False
    seen.add(key)
    return True

dataset = dataset.filter(unique_question, load_from_cache_file=False)
print(f"Unique nonempty questions: {len(dataset)}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

Filter:   0%|          | 0/68912 [00:00<?, ? examples/s]

Unique nonempty questions: 58485


## Load the tokenizer and create conversations

The tokenizer converts text into model tokens and supplies Llama's conversation template. Its EOS token is also used as the padding token, avoiding a vocabulary change.

`format_example` joins the instruction and optional input into the user's message. It places the reference answer in a separate assistant completion. The resulting `prompt` and `completion` columns let TRL apply the model's chat template and train only on completion tokens. Keeping the question separate also lets evaluation ask the model a question without accidentally showing the reference answer. `map` adds these columns while retaining the raw fields for later reports.


In [7]:
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    question = example["instruction"].strip()
    if example["input"].strip():
        question += "\n\n" + example["input"].strip()
    return {
        "prompt": [{"role": "user", "content": question}],
        "completion": [{"role": "assistant", "content": example["output"].strip()}],
    }

dataset = dataset.map(format_example)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/58485 [00:00<?, ? examples/s]

## Create independent training, validation and test splits

The first split reserves 20% of the deduplicated data. The second divides that held-out pool equally, giving approximately 80% training, 10% validation and 10% final testing. The same seed makes these selections repeatable for an unchanged dataset.

Training changes the adapter weights. Validation measures loss during training and selects the best checkpoint. Final testing produces the reported comparison only after training. Keeping the final test separate avoids reporting results on the same examples used to choose the checkpoint. If you repeatedly tune settings based on these test scores, reserve a fresh test set for a final assessment.


In [8]:
# Final test data is never used to select checkpoints.
split = dataset.train_test_split(test_size=0.2, seed=SEED)
heldout = split["test"].train_test_split(test_size=0.5, seed=SEED)
train_dataset = split["train"]
eval_dataset = heldout["train"]
test_dataset = heldout["test"]

## Choose manageable sample sizes

Each split is shuffled with the fixed seed, then capped at 3,000 training examples, 200 validation examples and 50 final test questions. `min` prevents selecting indices beyond the available rows if the dataset becomes smaller.

These defaults make a trial practical on a Colab GPU. Fifty questions provide an exploratory result, not a stable benchmark. Increase the test cap for stronger evidence, at the cost of generating an answer twice for every additional question. Change the caps before training/evaluation and rerun the dependent cells in order. The printed counts are the actual sample sizes used in this run.


In [9]:
train_dataset = train_dataset.shuffle(seed=SEED).select(range(min(3000, len(train_dataset))))
eval_dataset = eval_dataset.shuffle(seed=SEED).select(range(min(200, len(eval_dataset))))
raw_test = test_dataset.shuffle(seed=SEED).select(range(min(50, len(test_dataset))))
print(f"Train: {len(train_dataset)}; validation: {len(eval_dataset)}; final test: {len(raw_test)}")

Train: 3000; validation: 200; final test: 50


## Display the active GPU

This diagnostic prints whether CUDA is available and the name of device 0. It should agree with the earlier hardware check. It does not load another model or change training settings.

If you switch the runtime type, Colab starts a new environment: rerun setup and model loading rather than continuing with variables from the old session. This notebook uses one GPU.


In [10]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


## Load Llama with 4-bit quantization

`BitsAndBytesConfig` stores the large pretrained weights in 4-bit NF4 format. Double quantization also compresses the quantization constants. The selected FP16/BF16 compute dtype controls arithmetic; it does not mean that all model weights are stored at that precision.

The explicit device map places the model on the current CUDA GPU. `use_cache=False` disables the generation cache during training, where gradient checkpointing is used to save memory. Quantization reduces memory requirements but does not guarantee every sequence length or batch size fits. The model still requires gated Hugging Face access and a substantial first download.


In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": torch.cuda.current_device()},
)
model.config.use_cache = False
print("Model loaded successfully!")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!


## Inspect the model structure

Printing the model shows its transformer blocks, attention layers and projection modules. The module names explain the LoRA targets configured below: attention uses `q_proj`, `k_proj`, `v_proj` and `o_proj`, while the feed-forward network uses `gate_proj`, `up_proj` and `down_proj`.

This is an inspection step, not an accuracy measurement. A successful model print establishes that the object loaded; it does not establish that training or inference will work.


In [12]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

## Prepare the quantized model for training

`prepare_model_for_kbit_training` applies PEFT's preparation for training around quantized weights. It freezes pretrained parameters and makes the precision/gradient handling adjustments required by this workflow. The next cell adds trainable adapters.

QLoRA does not directly optimize the original 4-bit weights. It learns small adapter matrices while the pretrained model remains frozen. This is what makes adaptation of a multi-billion-parameter model practical on a smaller GPU.


In [13]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

## Attach the LoRA adapters

LoRA adds a low-rank update to each selected projection. Rank `r=16` controls the size of the learned matrices; `lora_alpha=32` controls their scaling. Dropout of 0.05 regularizes the adapter path during training, and `bias="none"` leaves bias parameters outside the adapter training scheme.

The targets cover attention and feed-forward projections. `get_peft_model` wraps the base model with these adapters, and `print_trainable_parameters` reports the actual trainable count and percentage. These counts describe training cost, not model quality. Changing the rank or target modules requires rebuilding the model and retraining.


In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## Configure training and checkpoint selection

One epoch processes the selected training examples once. A per-device batch size of one and eight gradient accumulation steps give an effective batch of eight examples on this single GPU. Gradient checkpointing saves memory by recomputing intermediate activations during backpropagation; the paged 8-bit AdamW optimizer reduces optimizer memory.

The learning rate is `2e-4`, with 11 warmup steps and a cosine schedule. Logs appear every 10 optimizer steps; validation and checkpoint saves happen every 50. The trainer retains up to two checkpoints and reloads the one with the lowest validation loss. With a tiny custom training set, the run may finish before the first scheduled evaluation.

Sequences are capped at 512 tokens, so long examples are truncated. Packing is disabled. `completion_only_loss=True` excludes prompt tokens from training loss. `<|eot_id|>` marks Llama's end of turn. FP16/BF16 follows the hardware check, and external experiment reporting is disabled.


In [15]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./llama3-finance-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=11,
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    bf16=use_bf16, fp16=not use_bf16,
    max_length=512, packing=False,
    completion_only_loss=True,
    eos_token="<|eot_id|>",
    seed=SEED, data_seed=SEED, report_to="none",
)

## Run supervised fine-tuning

`SFTTrainer` combines the PEFT model, configuration, tokenizer and datasets. Only the `prompt` and `completion` columns are passed, allowing TRL to format the conversations and construct completion masks without ambiguous extra text fields.

`trainer.train()` performs forward passes, computes answer-completion loss, backpropagates through the adapters and periodically validates/saves. The original pretrained weights stay frozen. The selected best checkpoint is loaded when training completes. Training loss and validation loss describe prediction likelihood; neither should be presented as a percentage of financially correct generated answers.


In [16]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_dataset.select_columns(["prompt", "completion"]),
    eval_dataset=eval_dataset.select_columns(["prompt", "completion"]),
    processing_class=tokenizer,
)
trainer.train()

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.413200,1.264100,2.262371,52787.000000,0.677455
100,1.496800,1.244455,2.281867,102489.000000,0.676772
150,1.762200,1.226616,2.283484,153902.000000,0.683656


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.413200,1.264100,2.262371,52787.000000,0.677455
100,1.496800,1.244455,2.281867,102489.000000,0.676772
150,1.762200,1.226616,2.283484,153902.000000,0.683656
200,1.597000,1.222975,2.134778,203016.000000,0.679855
250,1.571600,1.217082,2.149539,250882.000000,0.681901
300,1.625400,1.214769,2.159388,301274.000000,0.683460
350,1.468700,1.214144,2.149250,351412.000000,0.683200


TrainOutput(global_step=375, training_loss=1.5957505811055501, metrics={'train_runtime': 7004.9483, 'train_samples_per_second': 0.428, 'train_steps_per_second': 0.054, 'total_flos': 6378174778865664.0, 'train_loss': 1.5957505811055501, 'epoch': 1.0})

## Save the trained adapter and tokenizer

`save_model` writes the PEFT adapter and its configuration into `llama3-finance-lora-final`. `save_pretrained` stores the tokenizer files beside it. These files are saved inside the current Colab session; the download cell at the end copies them to your computer.

This is an adapter checkpoint, not a standalone copy of the full pretrained model. Loading it later still requires the compatible base model and access to that model. Save/download the results before Colab disconnects or deletes its temporary filesystem.


In [19]:
trainer.save_model("./llama3-finance-lora-final")
tokenizer.save_pretrained("./llama3-finance-lora-final")

('./llama3-finance-lora-final/tokenizer_config.json',
 './llama3-finance-lora-final/special_tokens_map.json',
 './llama3-finance-lora-final/chat_template.jinja',
 './llama3-finance-lora-final/tokenizer.json')

## Define the evaluation measurements

The model switches to evaluation mode, disabling training dropout, and re-enables the generation cache. Greedy decoding (`do_sample=False`) makes answers repeatable for a fixed model/runtime. `generate_answer` supplies only the user prompt and decodes only newly generated tokens, up to 256 tokens.

`normalize_answer` lowercases and collapses whitespace. It deliberately preserves decimal points, minus signs and currency symbols so `1.5`, `15`, `-5` and `5` remain distinct. `answer_scores` computes exact-match accuracy and mean token-overlap F1. F1 counts repeated words only up to their reference frequency; it is not a fact checker.

`answer_likelihood` masks the prompt with label `-100`, then scores reference-answer tokens within a 1,024-token window. Causal predictions are shifted by one position. Loss is weighted by the number of scored tokens across examples; perplexity is its exponential. Next-token accuracy checks those same positions while providing the correct preceding reference tokens, so it is **teacher-forced accuracy**, not generated-answer accuracy. Template end-of-turn tokens are included in answer likelihood. Truncated and skipped counts expose examples that do not fit the window.


In [17]:
import math
from collections import Counter
from contextlib import nullcontext

# All final metrics use the same held-out examples, prompt, and decoding settings.
model = trainer.model
model.eval()
model.config.use_cache = True
MAX_TEST_LENGTH = 1024
MAX_NEW_TOKENS = 256

def normalize_answer(text):
    # Preserve punctuation, signs and decimal points: finance numbers must stay distinct.
    return " ".join(text.lower().split())

def answer_scores(predictions, references):
    if not references or len(predictions) != len(references):
        raise ValueError("Need equal, nonempty prediction/reference lists")
    exact, f1 = 0, 0.0
    for prediction, reference in zip(predictions, references):
        a, b = normalize_answer(prediction), normalize_answer(reference)
        exact += a == b
        at, bt = a.split(), b.split()
        overlap = sum((Counter(at) & Counter(bt)).values())
        f1 += 2 * overlap / (len(at) + len(bt)) if at or bt else 1.0
    return {"exact_match_accuracy_pct": 100 * exact / len(references),
            "answer_token_f1_pct": 100 * f1 / len(references)}

def generate_answer(example):
    prompt = tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                 use_cache=True)
    return tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def answer_likelihood(examples):
    loss_sum, count, correct, skipped, truncated = 0.0, 0, 0, 0, 0
    for example in examples:
        prompt = tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
        full = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Chat template prompt is not a prefix of the full conversation")
        if len(full_ids) > MAX_TEST_LENGTH:
            truncated += 1
        ids = full_ids[:MAX_TEST_LENGTH]
        if len(prompt_ids) >= len(ids):
            skipped += 1
            continue
        inputs = torch.tensor([ids], device=model.device)
        labels = inputs.clone()
        labels[:, :len(prompt_ids)] = -100
        with torch.inference_mode():
            out = model(input_ids=inputs, labels=labels)
        targets = labels[:, 1:]
        mask = targets != -100
        tokens = int(mask.sum().item())
        loss_sum += out.loss.item() * tokens
        correct += int(((out.logits[:, :-1].argmax(-1) == targets) & mask).sum().item())
        count += tokens
    if not count:
        raise ValueError("No answer tokens fit in the test context window")
    loss = loss_sum / count
    return {"answer_loss": loss, "answer_perplexity": math.exp(loss),
            "next_token_accuracy_pct": 100 * correct / count,
            "evaluated_answer_tokens": count, "skipped_examples": skipped,
            "truncated_examples": truncated}

## Evaluate the base model and the fine-tuned model

The loop uses the same final test rows and reference answers for both models. `disable_adapter()` temporarily turns off LoRA to measure the quantized base model, and its context manager restores adapter behavior afterward. This avoids allocating a second full model on the T4.

For each model, the cell generates answers and computes exact match, token F1, answer loss/perplexity and next-token accuracy. Evaluate adds corpus BLEU and ROUGE scores. BLEU/ROUGE are multiplied by 100 for display on a percentage-style scale; this does not make them factual accuracy. `results` stores scores and `answers` stores generations. This cell can take time because both models answer every question. Only a completed run supplies fresh measured numbers.


In [22]:
import json
import evaluate
import torch
from contextlib import nullcontext

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

references = [e["output"].strip() for e in raw_test]
results, answers = {}, {}

model.eval()

for name in ("base", "fine_tuned"):
    # Disable LoRA temporarily to evaluate the base model.
    context = model.disable_adapter() if name == "base" else nullcontext()

    with context:
        # Handle mixed Float32 and FP16/BF16 layers during evaluation.
        with torch.inference_mode(), torch.autocast(
            device_type="cuda",
            dtype=compute_dtype,
        ):
            predictions = [generate_answer(e) for e in raw_test]
            scores = answer_scores(predictions, references)
            scores.update(answer_likelihood(raw_test))

    scores["bleu_pct"] = 100 * bleu.compute(
        predictions=predictions,
        references=[[reference] for reference in references],
    )["bleu"]

    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
    )
    scores.update({
        f"{metric}_pct": 100 * value
        for metric, value in rouge_scores.items()
    })

    results[name] = scores
    answers[name] = predictions

    print(name)
    print(json.dumps(scores, indent=2))

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


base
{
  "exact_match_accuracy_pct": 0.0,
  "answer_token_f1_pct": 21.031611767686762,
  "answer_loss": 2.388633197614069,
  "answer_perplexity": 10.898587542854067,
  "next_token_accuracy_pct": 49.40184883088635,
  "evaluated_answer_tokens": 3678,
  "skipped_examples": 0,
  "truncated_examples": 1,
  "bleu_pct": 3.6122010976235464,
  "rouge1_pct": 25.288046314245182,
  "rouge2_pct": 8.829624215263111,
  "rougeL_pct": 18.36664797073852,
  "rougeLsum_pct": 20.061497753406204
}
fine_tuned
{
  "exact_match_accuracy_pct": 2.0,
  "answer_token_f1_pct": 39.175486804196176,
  "answer_loss": 1.9390294102019043,
  "answer_perplexity": 6.9520001546042325,
  "next_token_accuracy_pct": 55.16585100598151,
  "evaluated_answer_tokens": 3678,
  "skipped_examples": 0,
  "truncated_examples": 1,
  "bleu_pct": 7.296113753586788,
  "rouge1_pct": 44.477202255993866,
  "rouge2_pct": 22.925683112727054,
  "rougeL_pct": 37.39684215236595,
  "rougeLsum_pct": 38.060540737117655
}


## Display and interpret the comparison table

Pandas turns the two score dictionaries into columns and adds `fine_tuned - base` as the change. Positive changes are desirable for exact match, F1, next-token accuracy, BLEU and ROUGE; negative changes are desirable for loss and perplexity.

For percentage metrics, a rise from 40 to 45 means **+5 percentage points**, not a 5% relative improvement. Evaluated token counts and truncation/skipping counts describe coverage rather than quality. A higher token accuracy or lower perplexity alone does not prove more accurate financial advice. Read the generated answers before drawing that conclusion.


In [23]:
import pandas as pd
from IPython.display import display
comparison = pd.DataFrame(results)
comparison["change"] = comparison["fine_tuned"] - comparison["base"]
display(comparison.round(4))
print("Accuracy/F1/ROUGE/BLEU changes are percentage points. Lower loss/perplexity is better.")
print("Next-token accuracy uses the correct preceding answer tokens; it is not factual QA accuracy.")
print("Exact match is strict answer equality after lowercasing and whitespace normalization.")

,base,fine_tuned,change
exact_match_accuracy_pct,0.0000,2.0000,2.0000
answer_token_f1_pct,21.0316,39.1755,18.1439
answer_loss,2.3886,1.9390,-0.4496
answer_perplexity,10.8986,6.9520,-3.9466
next_token_accuracy_pct,49.4018,55.1659,5.7640
evaluated_answer_tokens,3678.0000,3678.0000,0.0000
skipped_examples,0.0000,0.0000,0.0000
truncated_examples,1.0000,1.0000,0.0000
bleu_pct,3.6122,7.2961,3.6839
rouge1_pct,25.2880,44.4772,19.1892


Accuracy/F1/ROUGE/BLEU changes are percentage points. Lower loss/perplexity is better.
Next-token accuracy uses the correct preceding answer tokens; it is not factual QA accuracy.
Exact match is strict answer equality after lowercasing and whitespace normalization.


## Export scores and the evidence behind them

The cell creates `evaluation_results`, writes a structured `metrics.json` report and exports the comparison table as `comparison.csv`. The JSON includes the model/dataset names, seed, sample counts, evaluation limits, package versions, reference answers and both generated answers for each question.

This lets you inspect how the scores were obtained and compare later runs with their settings. Package versions help diagnose environment differences; they do not guarantee bit-for-bit reproduction or pin a changing upstream dataset/model revision. The exported files contain dataset questions and model outputs, so inspect them before sharing them.


In [24]:
# Save measured scores, provenance, and generated answers for review.
from pathlib import Path
import importlib.metadata
output = Path("evaluation_results")
output.mkdir(exist_ok=True)
report = {
    "model_id": model_id, "dataset": "gbharti/finance-alpaca", "seed": SEED,
    "train_examples": len(train_dataset), "validation_examples": len(eval_dataset),
    "test_examples": len(raw_test), "max_test_length": MAX_TEST_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "versions": {p: importlib.metadata.version(p) for p in ["torch", "transformers", "trl", "peft", "datasets"]},
    "results": results,
    "examples": [{"instruction": e["instruction"], "input": e["input"],
                  "reference": references[i], "base": answers["base"][i],
                  "fine_tuned": answers["fine_tuned"][i]} for i, e in enumerate(raw_test)],
}
(output / "metrics.json").write_text(json.dumps(report, indent=2))
comparison.to_csv(output / "comparison.csv")
print(f"Saved results to {output.resolve()}")

Saved results to /content/evaluation_results


## Reading the numbers

Exact-match accuracy and next-token accuracy are different measurements. Token F1 and ROUGE measure answer overlap, not financial correctness. A 50-question sample is exploratory; increase the test size for stronger evidence. The test set is separate from checkpoint validation. Inspect skipped/truncated counts alongside likelihood metrics.

## Optionally increase evaluation coverage

This comment-only cell reminds you that `raw_test` controls the final evaluation size. Change its cap in the sample-selection cell, rebuild `raw_test`, and rerun the evaluation definitions, scoring, table and export cells.

Changing only the final test sample size does not require retraining an unchanged adapter. Keep training settings and the validation split fixed when comparing test sample sizes. Record the new count with the resulting metrics.


In [25]:
# For a larger final evaluation, increase raw_test above and rerun the evaluation cells.

## Inspect individual answers

The loop prints up to three questions, base answers, fine-tuned answers and references. `min` makes this work even if fewer than three test questions are available.

Use these examples to spot truncated answers, incorrect quantities, unsupported claims and valid paraphrases that exact match penalizes. The full input context and all evaluated answers are included in `metrics.json`. Three examples are a spot-check, not evidence that every answer is correct.


In [26]:
for i in range(min(3, len(raw_test))):
    print(f"Question: {raw_test[i]['instruction']}")
    print(f"Base: {answers['base'][i]}")
    print(f"Fine-tuned: {answers['fine_tuned'][i]}")
    print(f"Reference: {references[i]}\n")

Question: Latest China devaluation (24/08/2015) and the affect on house prices in UK
Base: I couldn't find any information on a specific "latest China devaluation" on August 24, 2015, that had a direct impact on house prices in the UK.

However, I can provide some context on the Chinese devaluation in 2015 and its potential effects on global markets, including the UK.

In August 2015, China's central bank, the People's Bank of China (PBOC), devalued the Chinese yuan (RMB) by allowing it to float more freely on the foreign exchange market. This move was seen as a response to China's economic slowdown and efforts to stimulate economic growth.

The devaluation of the Chinese yuan had several effects on global markets:

1. **Currency markets:** The devaluation of the Chinese yuan led to a decline in the value of the yuan against other major currencies, including the US dollar, the euro, and the pound sterling. This made Chinese exports more expensive and reduced demand for them.
2. **Stock

## Download the adapter and evaluation report

The zip command packages the final adapter/tokenizer folder together with the evaluation report folder. Colab's `files.download` starts a browser download of that archive.

Run this after both saving the model and exporting evaluation results. The archive is the reusable output of this run; confirm the download completes before ending the Colab session. It contains adapter weights and reports, while the base Llama weights remain a separate dependency.


In [27]:
!zip -rq llama3-finance-results.zip ./llama3-finance-lora-final ./evaluation_results
from google.colab import files
files.download("llama3-finance-results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>